In [0]:
import data.utilities.common as Commonlib

import pyspark.sql.functions as F

from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU
from data.utilities.publishers import SnowflakeExternalTable
from pyspark.sql.types import TimestampType

In [0]:
CONFIG_BASE_PATH = "configuration/"

config_entries = Commonlib.list_files(CONFIG_BASE_PATH, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "configuration file")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}/{config_file}"

config = Commonlib.get_object_config(config_file_path)

In [0]:
# Instantiate medallion object
medallion = Medallion(config)

In [0]:
SOURCE_TABLES = {
    # silver source tables
    "tgsb": MTU.get_fully_qualified_table_name("sap", "ab1", "tgsb"),
    "tgsbt": MTU.get_fully_qualified_table_name("sap", "ab1", "tgsbt"),
}

In [0]:
# mapping from silver tables to gold table
col_mappings = {
    "bus_area_cd": F.col("tgsb.gsber"),
    "bus_area_nm": F.col("tgsbt.gtext"),
}

In [0]:
try:
    # Base DF for join
    left_df = spark.read.table(SOURCE_TABLES["tgsb"]).alias("tgsb")
    # other table to join to base
    tgsbt_df = spark.read.table(SOURCE_TABLES["tgsbt"]).alias("tgsbt")

    # join silver table
    left_df = left_df.join(
        other=tgsbt_df,
        on=(F.col("tgsb.gsber") == F.col("tgsbt.gsber")) & (F.col("tgsb.mandt") == F.col("tgsbt.mandt")),
        how="left",
    )

    # Transform and select final columns
    medallion.df = (
        left_df.withColumns(col_mappings)
        .select(*col_mappings.keys())
        .withColumn("__created_tsp", F.lit(medallion.start_datetime).cast(TimestampType()))
    )

except Exception as e:
    medallion.logger.error(f"Error processing business_area source table reads/joins. Error message: {e}")
    raise

In [0]:
# write gold table
try:
    medallion.write_to_delta(load_type="overwrite", layer="gold")
except Exception as e:
    medallion.logger.error(f"Error writing business_area table to gold layer. Error message: {e}")
    raise

In [0]:
# publish to external stage - snowflake
try:
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(medallion.catalog, medallion.schema, medallion.name)
        pub.publish_ext_table()

except Exception as e:
    medallion.logger.error(e)
    raise

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config["name"],
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
        config.get("comparison_check", {}).get("treat_nulls"),
    )
except KeyError as e:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )